# Conformal Triage — Phase 5: what Section 4 promises and Section 5 did not yet report

Runs on `conformal-triage/emb/` and `conformal-triage/results/head_weights.npz` in your Drive.
**CPU is enough**, ~20–30 min. It does not touch any Phase 3–4 artefact: everything new is
written with the prefix `f5_` into `conformal-triage/results/`.

What it computes, and which promise of the paper it answers:

| Cell | Output | Promise |
|---|---|---|
| 4 | `f5_alpha_curve.csv`, `f5_fig_alpha_curve.pdf` | 4.5: sensitivity–referral trade-off in α (Mondrian, marginal, softmax threshold) |
| 5 | `f5_benign_by_group.csv` | 4.3, caveat 2: benign coverage by phototype under the pooled threshold |
| 6 | `f5_hiba_modality.csv`, `f5_mixed_lesion_ablation.csv` | 4.2: results by modality; 3.4: lesions with dermoscopy + clinical images |
| 7 | `f5_intervals.csv` | 3.4 / 4.5: patient-level bootstrap and exact binomial intervals |
| 8 | `f5_padhead_results.csv`, `f5_padhead_report.txt` | Section 6: target-domain head (PAD-only) against the primary probe |
| 9 | `f5_perclass_f1_auc.csv`, `f5_head_details.txt` | 4.5: per-class F1 and AUC; 4.6: head configuration |
| 10 | `f5_resumen.txt` | Numbers to paste into Section 5 |

**About the draws.** This notebook reproduces the Phase-3 draws exactly: same seeds
(`SEED*1000 + d`), same rng consumption order (HIBA partition, HIBA de-clustering, one marginal
de-clustering per alpha, PAD partition, PAD de-clustering), the fixed stratum iteration order of
the patched notebooks 02/03, and the same sklearn code path for the probabilities. Cell 3 checks
partitions and per-stratum coverage against `results_draws.csv` for every draw and alpha and
prints 100/100 for both cohorts when the reproduction is exact. Every comparison in this notebook
is within identical draws, and the primary configuration is recomputed here as the reference.

In [ ]:
# 1) Setup: Drive (or a local folder), embeddings, metadata, Phase-3 head
SEED = 2026
import os, json, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/conformal-triage'
except Exception:
    BASE = os.environ.get('CT_BASE', os.path.expanduser('~/conformal-triage'))
SMOKE = os.environ.get('CT_SMOKE') == '1'   # only for the synthetic-data smoke test
RES = f'{BASE}/results'; os.makedirs(RES, exist_ok=True)
def out(name): return f'{RES}/f5_{name}'

d = np.load(f'{BASE}/emb/hiba_emb.npz'); Eh = dict(zip(d['ids'], d['features'].astype(np.float32)))
d = np.load(f'{BASE}/emb/pad_emb.npz');  Ep = dict(zip(d['ids'], d['features'].astype(np.float32)))
Xi, yi = [], []
for k in range(3):
    d = np.load(f'{BASE}/emb/isic2019_emb_part{k}.npz')
    Xi.append(d['features'].astype(np.float32)); yi.append(d['labels'])
Xisic = np.concatenate(Xi); yisic = np.concatenate(yi).astype(int)
Hm = pd.read_csv(f'{BASE}/emb/hiba_isic_metadata.csv')
Pm = pd.read_csv(f'{BASE}/emb/pad_isic_metadata.csv')
w = np.load(f'{BASE}/results/head_weights.npz')
COEF, INTC, SC_M, SC_S = w['coef'], w['intercept'], w['scaler_mean'], w['scaler_scale']
# Rebuild the Phase-3 sklearn objects so that probabilities follow the SAME numerical code path
# as notebook 02 (float32 standardisation, float64 decision function). A hand-written softmax in
# float64 differs from sklearn at ~1e-7 near score 1.0, which is enough to move the alpha=0.05
# benign threshold on some draws.
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
_sc = StandardScaler(); _sc.mean_, _sc.scale_, _sc.var_ = SC_M, SC_S, SC_S ** 2
_sc.n_features_in_ = len(SC_M); _sc.n_samples_seen_ = 1
_head = LogisticRegression(); _head.coef_, _head.intercept_ = COEF, INTC
_head.classes_ = np.arange(COEF.shape[0]); _head.n_features_in_ = len(SC_M)
def probs_of(X):
    return _head.predict_proba(_sc.transform(X))
print(f'loaded: Phase-3 head | {len(Eh)} HIBA embeddings | {len(Ep)} PAD embeddings | ISIC {Xisic.shape}')

In [ ]:
# 2) Pipeline functions (identical to Phases 3/4 except for the fixed stratum iteration order)
CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
MAL = {'MEL','BCC','SCC','AK'}
ROMAN = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6}
HIBA_MAP = {'melanoma':'MEL','nevus':'NV','basal cell carcinoma':'BCC',
            'squamous cell carcinoma':'SCC','actinic keratosis':'AK',
            'seborrheic keratosis':'BKL','solar lentigo':'BKL',
            'lichenoid keratosis':'BKL','dermatofibroma':'DF','vascular lesion':'VASC'}
DX3_MAP = {'Melanoma, NOS':'MEL','Melanoma':'MEL','Nevus':'NV',
           'Basal cell carcinoma':'BCC','Squamous cell carcinoma, NOS':'SCC',
           'Squamous cell carcinoma':'SCC','Solar or actinic keratosis':'AK',
           'Seborrheic keratosis':'BKL','Solar lentigo':'BKL',
           'Lichen planus like keratosis':'BKL','Lichenoid keratosis':'BKL',
           'Dermatofibroma':'DF','Hemangioma':'VASC','Angioma':'VASC',
           'Vascular lesion':'VASC',
           'Benign soft tissue proliferations - Vascular':'VASC'}

def lesion_table(meta, cohort):
    df = meta.copy()
    if 'diagnosis' in df.columns and df['diagnosis'].notna().any():
        dx = df['diagnosis']
    else:
        d2 = df['diagnosis_2'] if 'diagnosis_2' in df.columns else pd.Series(np.nan, index=df.index)
        dx = df['diagnosis_3'].fillna(d2)
    df['cls'] = dx.map({**HIBA_MAP, **DX3_MAP})
    df['g'] = df['fitzpatrick_skin_type'].map(ROMAN)
    missing = sorted(dx[df['cls'].isna()].dropna().unique().tolist()) + (['(empty)'] if (df['cls'].isna() & dx.isna()).any() else [])
    assert not missing, f'{cohort}: unmapped diagnoses: {missing}'
    les = df.groupby('lesion_id').agg(cls=('cls','first'), patient=('patient_id','first'),
                                      g=('g','first'), imgs=('isic_id', lambda s: tuple(s))).reset_index()
    chk = df.groupby('lesion_id').agg(nc=('cls','nunique'), npat=('patient_id','nunique'))
    assert (chk.nc == 1).all() and (chk.npat == 1).all(), f'{cohort}: lesion with inconsistent class or patient'
    les['mal'] = les.cls.isin(MAL)
    return les

def membership(les):
    m = {}
    for pid, sub in les[les.g.notna()].groupby('patient'):
        s = set()
        for _, r in sub.iterrows(): s.add(('M', int(r.g)) if r.mal else ('B',))
        m[pid] = s
    return m

def floor_n(alpha): return int(np.ceil(1.0/alpha - 1e-9)) - 1

def pad_training_draw(les, rng, train_frac=0.4, cal_frac=0.6, alpha=0.05, max_tries=100):
    pf = les.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
    pgroup = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
    pats = np.array(sorted(pgroup)); grp = np.array([pgroup[p] for p in pats])
    mem = membership(les); enforced = [('M',1), ('M',2), ('M',3), ('B',)]; fl = floor_n(alpha)
    for _ in range(max_tries):
        tr = set()
        for g in np.unique(grp):
            idx = np.where(grp == g)[0]
            tr.update(pats[rng.choice(idx, int(round(train_frac*len(idx))), replace=False)])
        rem = [p for p in pats if p not in tr]
        if all(cal_frac*sum(1 for p in rem if s in mem.get(p, ())) >= fl for s in enforced):
            return sorted(tr), sorted(rem)
    raise RuntimeError('could not verify the training draw')

def aps_scores(P):
    order = np.argsort(-P, axis=1); cum = np.take_along_axis(P, order, 1).cumsum(1)
    S = np.empty_like(P); np.put_along_axis(S, order, cum, 1); return S

def qhat(scores, alpha):
    n = len(scores); k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    return np.inf if (n == 0 or k > n) else np.sort(scores)[k - 1]

def draw_cells(les, cal_pats, rng):
    """Per-stratum de-clustering, one lesion per patient. FIXED stratum order (key=str)."""
    cal = les[les.patient.isin(cal_pats)]
    mem_cal = membership(cal); ph = cal[cal.g.notna()]; cells = {}
    for s in sorted(set().union(*mem_cal.values()) if mem_cal else set(), key=str):
        rows = ph[(ph.mal & (ph.g == s[1]))] if s[0] == 'M' else ph[~ph.mal]
        cells[s] = np.array(sorted(rng.choice(sub.index.values) for _, sub in rows.groupby('patient')), dtype=int)
    return cal, cells

def marginal_picks(cal, rng):
    return np.array(sorted(rng.choice(sub.index.values) for _, sub in cal[cal.g.notna()].groupby('patient')), dtype=int)

MALc = [CLS.index(c) for c in CLS if c in MAL]

def mondrian_sets(les, S, cells, test, alpha):
    """Mondrian sets (one threshold per (group, pooled M) + one pooled B threshold) on the test set."""
    q = {s: (qhat(S[cells[s], [CLS.index(les.loc[i, 'cls']) for i in cells[s]]], alpha) if len(cells[s]) else np.inf) for s in cells}
    qB = q.get(('B',), np.inf); Stest = S[test.index.values]
    Ct = np.zeros((len(test), len(CLS)), bool)
    for c, name in enumerate(CLS):
        thr = np.array([q.get(('M', int(g)), np.inf) for g in test.g.values]) if name in MAL else np.full(len(test), qB)
        Ct[:, c] = Stest[:, c] <= thr
    return Ct, q

def marginal_sets(les, S, picks, test, alpha):
    qm = qhat(S[picks, [CLS.index(les.loc[i, 'cls']) for i in picks]], alpha)
    return S[test.index.values] <= qm, qm

def observe_of(Ct): return (Ct.sum(1) == 1) & ~Ct[:, MALc].any(1)

def test_of(les, test_pats): return les[les.patient.isin(test_pats) & les.g.notna()]

def ytrue_of(test): return np.array([CLS.index(c) for c in test.cls.values])

def softmax_rule(P, les, picks, test, alphas, T=None):
    """Phase-4 softmax-threshold rule (rule_eval), with the calibration picks already given."""
    if T == 'fit':
        from scipy.optimize import minimize_scalar
        y = np.array([CLS.index(c) for c in les.loc[picks, 'cls']]); lg0 = np.log(np.clip(P[picks], 1e-12, 1))
        def nll(t):
            z = lg0 / t; z = z - z.max(1, keepdims=True); pr = np.exp(z); pr /= pr.sum(1, keepdims=True)
            return -np.log(np.clip(pr[np.arange(len(y)), y], 1e-12, 1)).mean()
        T = float(minimize_scalar(nll, bounds=(0.25, 10), method='bounded').x)
    T = T or 1.0
    lg = np.log(np.clip(P, 1e-12, 1)) / T; lg -= lg.max(1, keepdims=True); Pt = np.exp(lg); Pt /= Pt.sum(1, keepdims=True)
    top, pred = Pt.max(1), Pt.argmax(1); pred_ben = ~np.isin(np.array(CLS)[pred], list(MAL))
    mal_c = les.mal.values[picks]; out = {}
    for a in alphas:
        tau = 1.01
        for t in np.concatenate([[0.0], np.sort(top[picks])]):
            o = pred_ben[picks] & (top[picks] >= t)
            if (o & mal_c).sum() / max(mal_c.sum(), 1) <= a: tau = t; break
        ti = test.index.values; out[a] = pred_ben[ti] & (top[ti] >= tau)
    return out

def pw_mean(patients, values):
    return float(pd.DataFrame({'p': patients, 'v': values}).groupby('p').v.mean().mean())

In [ ]:
# 3) Tables, training draw, scores; count checks and partition check against Phase 3
hles = lesion_table(Hm, 'hiba').reset_index(drop=True)
ples = lesion_table(Pm, 'pad').reset_index(drop=True)
if not SMOKE:
    assert len(hles) == 1246 and len(ples) == 1891, (len(hles), len(ples))
rng = np.random.default_rng(SEED)
tr, rem = pad_training_draw(ples, rng)
def lesion_X(les, E): return np.stack([np.mean([E[i] for i in t], 0) for t in les.imgs])
Ph, Pp = probs_of(lesion_X(hles, Eh)), probs_of(lesion_X(ples, Ep))
Sh, Sp = aps_scores(Ph), aps_scores(Pp)
hp = sorted(membership(hles))
ALPHAS = [0.05, 0.10, 0.15]
print(f'training draw reproduced: {len(tr)} PAD patients to the head | scores ready')

def partitions(d):
    """Same rng consumption as Phase 3: HIBA partition, HIBA de-clustering, PAD partition, PAD de-clustering."""
    r = np.random.default_rng(SEED * 1000 + d)
    cal_h = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    calh_df, cells_h = draw_cells(hles, cal_h, r)
    # notebook 02 draws the marginal de-clustering once per alpha (3 times) before moving on to
    # PAD; consume the rng identically, and keep the first set (the one used at alpha = 0.05)
    picks_h = [marginal_picks(calh_df, r) for _ in ALPHAS][0]
    cal_p = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    calp_df, cells_p = draw_cells(ples, cal_p, r)
    picks_p = [marginal_picks(calp_df, r) for _ in ALPHAS][0]
    return dict(hiba=(cal_h, set(hp) - cal_h, cells_h, picks_h), pad=(cal_p, set(rem) - cal_p, cells_p, picks_p))

# Reproduction check against Phase 3: partitions AND Mondrian coverage per stratum, draw and alpha
# must be identical to results_draws.csv. Anything below 100/100 means this notebook is not on the
# same draws as Table 7 and its results must be labelled accordingly.
try:
    R3 = pd.read_csv(f'{RES}/results_draws.csv')
    ok = {'hiba': 0, 'pad': 0}
    for d in range(100):
        P = partitions(d)
        for c, les, S in [('hiba', hles, Sh), ('pad', ples, Sp)]:
            cal, tp, cells, picks = P[c]; test = test_of(les, tp); y = ytrue_of(test); mal, g = test.mal.values, test.g.values
            same = True
            for a in ALPHAS:
                Cm, q = mondrian_sets(les, S, cells, test, a); cov = Cm[np.arange(len(test)), y]
                ref = R3[(R3.cohort == c) & (R3.draw == d) & (R3.alpha == a) & (R3.stratum.isin(['B', 'M1', 'M2', 'M3']))]
                for st, ncal, cv in zip(ref.stratum, ref.n_cal, ref.coverage):
                    s_ = ('B',) if st == 'B' else ('M', int(st[1]))
                    sel = ~mal if st == 'B' else (mal & (g == s_[1]))
                    same &= (len(cells.get(s_, [])) == ncal) and (sel.sum() > 0) and abs(float(cov[sel].mean()) - cv) < 1e-9
            ok[c] += same
    print(f"identical to results_draws.csv (partitions + coverage, all strata and alphas): HIBA {ok['hiba']}/100, PAD {ok['pad']}/100")
    if min(ok.values()) < 100: print('  WARNING: not on the same draws as Phase 3 -- see the note at the top before using these results')
except FileNotFoundError:
    print('results_draws.csv not in results/: skipping the reproduction check')

In [ ]:
# 4) Sensitivity–referral trade-off in alpha: pooled Mondrian, marginal, softmax threshold, temperature scaling
ALPHA_GRID = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]
ENFORCED = [('M',1), ('M',2), ('M',3), ('B',)]
rows = []
t0 = time.time()
for d in range(100):
    P = partitions(d)
    for c, les, S, Pr in [('hiba', hles, Sh, Ph), ('pad', ples, Sp, Pp)]:
        cal, tp, cells, picks = P[c]; test = test_of(les, tp); y = ytrue_of(test)
        mal, g, pats = test.mal.values, test.g.values, test.patient.values
        cert = mal & np.isin(g, [1, 2, 3]); ben = ~mal
        r_soft = softmax_rule(Pr, les, picks, test, ALPHA_GRID, T=None)
        r_temp = softmax_rule(Pr, les, picks, test, ALPHA_GRID, T='fit')
        for a in ALPHA_GRID:
            Cm, q = mondrian_sets(les, S, cells, test, a); Cg, qm = marginal_sets(les, S, picks, test, a)
            for scheme, obs, Cset, deg in [('mondrian', observe_of(Cm), Cm, any(np.isinf(q.get(s, np.inf)) for s in ENFORCED if s in cells)),
                                           ('marginal', observe_of(Cg), Cg, bool(np.isinf(qm))),
                                           ('softmax', r_soft[a], None, False), ('temp', r_temp[a], None, False)]:
                worst = max(float(obs[mal & (g == k)].mean()) for k in (1, 2, 3) if (mal & (g == k)).sum()) if cert.sum() else np.nan
                rows.append(dict(cohort=c, draw=d, alpha=a, scheme=scheme,
                                 miss_cert=float(obs[cert].mean()) if cert.sum() else np.nan,
                                 miss_all=float(obs[mal].mean()) if mal.sum() else np.nan,
                                 worst_group_miss=worst, any_group_over=bool(worst > a) if not np.isnan(worst) else False,
                                 release=float(obs[ben].mean()) if ben.sum() else np.nan,
                                 release_pw=pw_mean(pats[ben], obs[ben]) if ben.sum() else np.nan,
                                 set_size=float(Cset.sum(1).mean()) if Cset is not None else np.nan,
                                 coverage=float(Cset[np.arange(len(test)), y].mean()) if Cset is not None else np.nan,
                                 any_enforced_degenerate=deg, n_test=len(test), n_mal=int(mal.sum()), n_ben=int(ben.sum())))
    if d % 20 == 0: print(f'  draw {d}/100  ({time.time()-t0:.0f}s)')
AC = pd.DataFrame(rows); AC.to_csv(out('alpha_curve.csv'), index=False)
acm = AC.groupby(['cohort', 'scheme', 'alpha']).agg(miss_cert=('miss_cert', 'mean'), worst=('worst_group_miss', 'mean'),
        over_pct=('any_group_over', lambda s: 100*s.mean()), release=('release', 'mean'), release_pw=('release_pw', 'mean'),
        set_size=('set_size', 'mean'), deg_pct=('any_enforced_degenerate', lambda s: 100*s.mean())).round(4)
print(acm.xs('mondrian', level='scheme').to_string())

import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
sty = {'mondrian': ('-', 'Mondrian (certified)'), 'marginal': ('--', 'marginal'), 'softmax': (':', 'softmax threshold'), 'temp': ('-.', 'temperature scaling')}
col = {'hiba': 'C0', 'pad': 'C3'}
for c in ['hiba', 'pad']:
    for s, (ls, lab) in sty.items():
        m = acm.loc[(c, s)]
        ax[0].plot(m.index, 100*m.release, ls, color=col[c], label=f'{c.upper() if c=="hiba" else "PAD"} {lab}')
        ax[1].plot(m.index, 100*m.worst, ls, color=col[c])
ax[1].plot(ALPHA_GRID, [100*a for a in ALPHA_GRID], color='0.5', lw=0.8, label=r'budget $\alpha$')
ax[0].set_xlabel(r'requested error level $\alpha$'); ax[0].set_ylabel('benign lesions released to observe [%]')
ax[1].set_xlabel(r'requested error level $\alpha$'); ax[1].set_ylabel('worst certified-group miss rate [%]')
ax[0].set_title('(a)', loc='left'); ax[1].set_title('(b)', loc='left')
ax[0].legend(fontsize=6.5, ncol=2, frameon=False); ax[1].legend(fontsize=7, frameon=False)
for a_ in ax: a_.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(out('fig_alpha_curve.pdf')); fig.savefig(out('fig_alpha_curve.png'), dpi=150)
print('figure ->', out('fig_alpha_curve.pdf'))

In [ ]:
# 5) Benign coverage by phototype under the pooled B threshold (Section 4.3, caveat 2)
rows = []
for d in range(100):
    P = partitions(d)
    for c, les, S in [('hiba', hles, Sh), ('pad', ples, Sp)]:
        cal, tp, cells, picks = P[c]; test = test_of(les, tp); y = ytrue_of(test); ben = ~test.mal.values
        for a in ALPHAS:
            Cm, q = mondrian_sets(les, S, cells, test, a); cov = Cm[np.arange(len(test)), y]; obs = observe_of(Cm)
            for k in sorted(set(test.g.values.astype(int))):
                sel = ben & (test.g.values == k)
                if sel.sum():
                    rows.append(dict(cohort=c, draw=d, alpha=a, group=k, n_benign=int(sel.sum()),
                                     benign_coverage=float(cov[sel].mean()), benign_referral=float((~obs[sel]).mean()),
                                     benign_set_size=float(Cm[sel].sum(1).mean())))
BG = pd.DataFrame(rows); BG.to_csv(out('benign_by_group.csv'), index=False)
bgm = BG.groupby(['cohort', 'alpha', 'group']).agg(n=('n_benign', 'mean'), cov=('benign_coverage', 'mean'), cov_sd=('benign_coverage', 'std'),
                                                  referral=('benign_referral', 'mean'), set_size=('benign_set_size', 'mean')).round(3)
print(bgm.to_string())

In [ ]:
# 6) HIBA by modality, and ablation of the lesions carrying both dermoscopy and clinical images
mod_col = next((c for c in Hm.columns if 'image_type' in c.lower()), None)
assert mod_col is not None, f'no modality column found in the HIBA metadata. Columns: {list(Hm.columns)}'
is_derm = Hm[mod_col].astype(str).str.lower().str.startswith('dermo')
derm_ids = set(Hm.loc[is_derm, 'isic_id'])
def modality(imgs):
    dd = sum(i in derm_ids for i in imgs)
    return 'derm' if dd == len(imgs) else ('clin' if dd == 0 else 'both')
hles['mod'] = [modality(t) for t in hles.imgs]
print('HIBA lesions by modality:', hles['mod'].value_counts().to_dict(), '(paper: both=290, clin=55)')

# alternative scores: for mixed lesions, the embedding of the dermoscopic image ONLY
def lesion_X_derm(les, E):
    return np.stack([np.mean([E[i] for i in t if (i in derm_ids or not any(j in derm_ids for j in t))], 0) for t in les.imgs])
Sh_derm = aps_scores(probs_of(lesion_X_derm(hles, Eh)))

rows, rows2 = [], []
for d in range(100):
    P = partitions(d); cal, tp, cells, picks = P['hiba']; test = test_of(hles, tp); y = ytrue_of(test)
    mal, ben, mod = test.mal.values, ~test.mal.values, test['mod'].values
    for a in ALPHAS:
        for variant, S in [('averaged', Sh), ('derm_only_for_mixed', Sh_derm)]:
            Cm, q = mondrian_sets(hles, S, cells, test, a); cov = Cm[np.arange(len(test)), y]; obs = observe_of(Cm)
            for m in ['derm', 'clin', 'both', 'all']:
                sel = np.ones(len(test), bool) if m == 'all' else (mod == m)
                if sel.sum():
                    rows.append(dict(variant=variant, draw=d, alpha=a, modality=m, n=int(sel.sum()), n_mal=int((sel & mal).sum()),
                                     coverage=float(cov[sel].mean()), set_size=float(Cm[sel].sum(1).mean()),
                                     miss=float(obs[sel & mal].mean()) if (sel & mal).sum() else np.nan,
                                     release=float(obs[sel & ben].mean()) if (sel & ben).sum() else np.nan,
                                     referral=float((~obs[sel]).mean())))
MO = pd.DataFrame(rows)
MO[MO.variant == 'averaged'].drop(columns='variant').to_csv(out('hiba_modality.csv'), index=False)
MO.to_csv(out('mixed_lesion_ablation.csv'), index=False)
print('\nHIBA by modality (primary configuration, averaged embedding):')
print(MO[MO.variant == 'averaged'].groupby(['alpha', 'modality']).agg(n=('n', 'mean'), cov=('coverage', 'mean'), set_size=('set_size', 'mean'),
      miss=('miss', 'mean'), release=('release', 'mean')).round(3).to_string())
print('\nmixed lesions: averaged vs dermoscopy-only')
print(MO[MO.modality == 'both'].groupby(['alpha', 'variant']).agg(cov=('coverage', 'mean'), set_size=('set_size', 'mean'),
      release=('release', 'mean'), miss=('miss', 'mean')).round(3).to_string())

In [ ]:
# 7) Intervals: exact binomial (Clopper–Pearson) for the per-stratum miss, patient-level bootstrap for release and miss
from scipy.stats import beta as Beta
def clopper_pearson(k, n, level=0.95):
    if n == 0: return (np.nan, np.nan)
    lo = 0.0 if k == 0 else Beta.ppf((1-level)/2, k, n-k+1)
    hi = 1.0 if k == n else Beta.ppf(1-(1-level)/2, k+1, n-k)
    return float(lo), float(hi)

def cluster_boot(patients, values, rng, B=500):
    df = pd.DataFrame({'p': patients, 'v': values}); grp = df.groupby('p').v.agg(['sum', 'count'])
    P_ = len(grp); est = []
    for _ in range(B):
        ix = rng.integers(0, P_, P_); s = grp.iloc[ix]
        est.append(s['sum'].sum() / s['count'].sum())
    return float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

rows = []
for d in range(100):
    P = partitions(d); rb = np.random.default_rng((SEED + 11) * 1000 + d)
    for c, les, S in [('hiba', hles, Sh), ('pad', ples, Sp)]:
        cal, tp, cells, picks = P[c]; test = test_of(les, tp)
        mal, g, pats = test.mal.values, test.g.values, test.patient.values
        for a in ALPHAS:
            Cm, q = mondrian_sets(les, S, cells, test, a); obs = observe_of(Cm)
            for s in [('M', 1), ('M', 2), ('M', 3), ('B',)]:
                sel = (mal & (g == s[1])) if s[0] == 'M' else ~mal
                if not sel.sum(): continue
                k = int(obs[sel].sum()); n = int(sel.sum()); npat = len(set(pats[sel]))
                cp = clopper_pearson(k, n); bt = cluster_boot(pats[sel], obs[sel], rb)
                rows.append(dict(cohort=c, draw=d, alpha=a, stratum=f"{s[0]}{s[1] if s[0]=='M' else ''}", n_lesions=n, n_patients=npat,
                                 rate=k/n, cp_lo=cp[0], cp_hi=cp[1], boot_lo=bt[0], boot_hi=bt[1]))
IV = pd.DataFrame(rows); IV.to_csv(out('intervals.csv'), index=False)
ivm = IV.groupby(['cohort', 'stratum', 'alpha']).agg(n_les=('n_lesions', 'mean'), n_pat=('n_patients', 'mean'), rate=('rate', 'mean'),
        cp_lo=('cp_lo', 'mean'), cp_hi=('cp_hi', 'mean'), boot_lo=('boot_lo', 'mean'), boot_hi=('boot_hi', 'mean')).round(3)
print('miss (M*) / release (B): mean rate, mean exact binomial CI, mean patient-bootstrap CI')
print(ivm.to_string())

In [ ]:
# 8) Target-domain head: probe trained ONLY on the PAD training portion (same 550 patients)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
pad_tr = ples[ples.patient.isin(tr)]
Xpt = np.stack([Ep[i] for t in pad_tr.imgs for i in t])
ypt = np.array([CLS.index(r.cls) for _, r in pad_tr.iterrows() for _ in r.imgs])
sc3 = StandardScaler().fit(Xpt)
head3 = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc3.transform(Xpt), ypt)
present = list(head3.classes_)
def probs3(X):
    pr = head3.predict_proba(sc3.transform(X)); P8 = np.zeros((len(X), len(CLS)), np.float64)
    P8[:, present] = pr; return P8      # classes absent from PAD (DF, VASC) get probability 0
ba3 = balanced_accuracy_score(ypt, head3.predict(sc3.transform(Xpt)))
Ph3, Pp3 = probs3(lesion_X(hles, Eh)), probs3(lesion_X(ples, Ep))
Sh3, Sp3 = aps_scores(Ph3), aps_scores(Pp3)
rows, cm = [], []
for d in range(100):
    P = partitions(d)
    for c, les, S, Pr, Pr0 in [('pad', ples, Sp3, Pp3, Pp), ('hiba', hles, Sh3, Ph3, Ph)]:
        cal, tp, cells, picks = P[c]; test = test_of(les, tp); y = ytrue_of(test); mal, g, ben = test.mal.values, test.g.values, ~test.mal.values
        for a in ALPHAS:
            for head_name, SS in [('pad_only', S), ('primary', Sh if c == 'hiba' else Sp)]:
                Cm, q = mondrian_sets(les, SS, cells, test, a); cov = Cm[np.arange(len(test)), y]; obs = observe_of(Cm)
                for s in [('M', 1), ('M', 2), ('M', 3), ('B',)]:
                    sel = (mal & (g == s[1])) if s[0] == 'M' else ben
                    if not sel.sum(): continue
                    rows.append(dict(cohort=c, head=head_name, draw=d, alpha=a, stratum=f"{s[0]}{s[1] if s[0]=='M' else ''}",
                                     degenerate=bool(np.isinf(q.get(s, np.inf))), coverage=float(cov[sel].mean()),
                                     miss_rate=float(obs[sel].mean()) if s[0] == 'M' else np.nan,
                                     referral=float((~obs[sel]).mean()), set_size=float(Cm[sel].sum(1).mean()), n_test=int(sel.sum())))
        for head_name, PP in [('pad_only', Pr), ('primary', Pr0)]:
            pr = PP[test.index.values]; yhat = pr.argmax(1)
            aucs = [roc_auc_score((y == k).astype(int), pr[:, k]) for k in range(8) if 0 < (y == k).sum() < len(y)]
            cm.append(dict(cohort=c, head=head_name, draw=d, top1=float((yhat == y).mean()), bal_acc=float(balanced_accuracy_score(y, yhat)),
                           macro_f1=float(f1_score(y, yhat, average='macro', labels=np.arange(8), zero_division=0)), auc_ovr=float(np.mean(aucs))))
PH = pd.DataFrame(rows); PH.to_csv(out('padhead_results.csv'), index=False)
CM3 = pd.DataFrame(cm); CM3.to_csv(out('padhead_class_metrics.csv'), index=False)
phm = PH.groupby(['cohort', 'head', 'alpha', 'stratum']).agg(deg=('degenerate', 'mean'), cov=('coverage', 'mean'), miss=('miss_rate', 'mean'),
                                                             referral=('referral', 'mean'), set_size=('set_size', 'mean')).round(3)
print('PAD-only head vs primary, by stratum:'); print(phm.to_string())
print('\nclassification metrics:'); print(CM3.groupby(['cohort', 'head']).mean(numeric_only=True).drop(columns='draw').round(3).to_string())
open(out('padhead_report.txt'), 'w').write(
    f'seed={SEED}\nPAD-only head: LogisticRegression(lbfgs, L2, C=1.0, max_iter=1000, class_weight=balanced) on StandardScaler\n'
    f'training images: {len(ypt)} (PAD training portion, {len(tr)} patients)\nclasses present: {[CLS[i] for i in present]}\n'
    f'balanced accuracy (train): {ba3:.4f}\n')

In [ ]:
# 9) Per-class F1 and AUC (primary head, same draws) + head details for Section 4.6
import sklearn
rows = []
for d in range(100):
    P = partitions(d)
    for c, les, Pr in [('hiba', hles, Ph), ('pad', ples, Pp)]:
        cal, tp, cells, picks = P[c]; test = test_of(les, tp); y = ytrue_of(test); pr = Pr[test.index.values]; yhat = pr.argmax(1)
        f1s = f1_score(y, yhat, average=None, labels=np.arange(8), zero_division=0)
        for k, name in enumerate(CLS):
            pos = int((y == k).sum())
            rows.append(dict(cohort=c, draw=d, cls=name, n=pos, f1=float(f1s[k]) if pos else np.nan,
                             auc=float(roc_auc_score((y == k).astype(int), pr[:, k])) if 0 < pos < len(y) else np.nan))
PC = pd.DataFrame(rows); PC.to_csv(out('perclass_f1_auc.csv'), index=False)
print(PC.groupby(['cohort', 'cls']).agg(n=('n', 'mean'), f1=('f1', 'mean'), auc=('auc', 'mean')).round(3).to_string())
details = (f'Primary head (Phase 3, seed {SEED}): sklearn {sklearn.__version__} LogisticRegression, lbfgs solver, L2 penalty (C=1.0), '
           f'multinomial, max_iter=1000, class_weight="balanced" (class-balanced cross-entropy), no other regularisation, '
           f'on standardised 1024-d PanDerm ViT-L embeddings (StandardScaler fit on the training set). '
           f'No epochs or schedule: a convex lbfgs fit to convergence or 1000 iterations. '
           f'Training: full ISIC 2019 ({len(yisic)} images) + PAD training portion ({len(ypt)} images, {len(tr)} patients). '
           f'Lesion embedding = mean of its image embeddings.\n')
open(out('head_details.txt'), 'w').write(details); print(details)

In [ ]:
# 10) Summary for Section 5
L = []
L.append('== Curve in alpha (pooled Mondrian): benign release lesion / patient, certified-group miss, Pr(any enforced stratum degenerate)')
for c in ['hiba', 'pad']:
    m = acm.loc[(c, 'mondrian')]
    L.append(c + ': ' + '; '.join(f'a={a:.2f}: rel {100*r['release']:.1f}%/{100*r['release_pw']:.1f}%, miss {100*r['miss_cert']:.2f}%, deg {r['deg_pct']:.1f}%' for a, r in m.iterrows()))
L.append('== Smallest alpha with benign release >= 5% (lesion-level) under Mondrian:')
for c in ['hiba', 'pad']:
    m = acm.loc[(c, 'mondrian')]; a5 = m[m.release >= 0.05].index.min()
    L.append(f'  {c}: {a5 if not np.isnan(a5) else "none on the grid"}')
L.append('== Marginal / softmax / temp: % of draws with some certified group over budget')
for c in ['hiba', 'pad']:
    for s in ['marginal', 'softmax', 'temp']:
        m = acm.loc[(c, s)]; L.append(f'  {c} {s}: ' + ', '.join(f'a={a:.2f}: {r['over_pct']:.0f}%' for a, r in m.iterrows() if a in ALPHAS))
L.append('== Benign coverage by phototype under the pooled threshold (a=0.05 / 0.15)')
for c in ['hiba', 'pad']:
    for a in [0.05, 0.15]:
        m = bgm.loc[(c, a)]; L.append(f'  {c} a={a}: ' + ', '.join(f'g{k}: {r['cov']:.3f} (n={r['n']:.0f})' for k, r in m.iterrows()))
L.append('== HIBA by modality (a=0.05 / 0.15): coverage, |C|, release')
for a in [0.05, 0.15]:
    m = MO[(MO.variant == 'averaged') & (MO.alpha == a)].groupby('modality')[['coverage', 'set_size', 'release']].mean()
    L.append(f'  a={a}: ' + ', '.join(f'{k}: cov {r['coverage']:.3f} |C| {r['set_size']:.2f} rel {100*r['release']:.1f}%' for k, r in m.iterrows()))
L.append('== Mixed lesions: averaged vs dermoscopy-only (a=0.05): coverage / |C|')
m = MO[(MO.modality == 'both') & (MO.alpha == 0.05)].groupby('variant')[['coverage', 'set_size']].mean()
L.append('  ' + ', '.join(f'{k}: {r['coverage']:.3f} / {r['set_size']:.2f}' for k, r in m.iterrows()))
L.append('== Type I/III intervals (a=0.05): mean miss [mean CP] [mean bootstrap], n lesions / patients')
for c in ['hiba', 'pad']:
    for s in ['M1', 'M3']:
        r = ivm.loc[(c, s, 0.05)]; L.append(f'  {c} {s}: {r['rate']:.3f} [{r['cp_lo']:.3f},{r['cp_hi']:.3f}] [{r['boot_lo']:.3f},{r['boot_hi']:.3f}] n={r['n_les']:.0f}/{r['n_pat']:.0f}')
L.append('== PAD-only head vs primary on PAD: |C| and benign release (a=0.05 / 0.15), and metrics')
for a in [0.05, 0.15]:
    for h in ['primary', 'pad_only']:
        r = phm.loc[('pad', h, a, 'B')]; L.append(f'  a={a} {h}: |C| benign {r['set_size']:.2f}, release {100*(1-r['referral']):.1f}%, deg {100*r['deg']:.0f}%')
L.append('  ' + CM3[CM3.cohort == 'pad'].groupby('head').mean(numeric_only=True).drop(columns='draw').round(3).to_string().replace('\n', '\n  '))
txt = '\n'.join(L); open(out('resumen.txt'), 'w').write(txt); print(txt)
print('\nDONE. New files in results/: ' + ', '.join(sorted(f for f in os.listdir(RES) if f.startswith('f5_'))))